# Lab 4 — Coordinate, Validate, Escalate
## Agentic Integration Skeleton

**Mission:** A planner has two recruiters and 16 field hours next week. Coordinate the earlier capabilities into a proposed plan for human review.

The agent may call tools, but it may not invent evidence, exceed the resource budget, or operationalize its own plan.

*This is a first-pass scaffold. It intentionally uses deterministic mock tools before any live model-driven orchestration is added.*

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

In [ ]:
from IPython.display import display, Markdown

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

In [ ]:
import pandas as pd

HOURS_AVAILABLE = 16
HUMAN_APPROVAL_REQUIRED = True

## 1. Define narrow tools

Tools should have one clear job, typed inputs, predictable outputs, and explicit failure states.

In [ ]:
def rank_schools(top_k=3):
    return [
        {"school": "Jefferson High", "score": .91, "historical_events": 19},
        {"school": "Washington High", "score": .83, "historical_events": 17},
        {"school": "North County Tech", "score": .80, "historical_events": 14},
    ][:top_k]

def recommend_actions(school):
    catalog = {
        "Jefferson High": {"action": "Mechanical Careers Demo", "score": .72, "evidence": "predicted"},
        "Washington High": {"action": "Cyber Careers Event", "score": .70, "evidence": "observed"},
        "North County Tech": {"action": "STEM Careers Presentation", "score": .66, "evidence": "observed"},
    }
    return catalog.get(school)

def retrieve_approved_information(school, action):
    if school == "Jefferson High" and action == "Mechanical Careers Demo":
        return {"source_ids": ["SCHOOL_PROFILE_2026_08", "MECH_PLAYBOOK_V3"], "sources_found": 2}
    return {"source_ids": [], "sources_found": 0}

def estimate_hours(school, action):
    return {"Mechanical Careers Demo": 6, "Cyber Careers Event": 5,
            "STEM Careers Presentation": 4}.get(action, 4)

## 2. Add validation gates before orchestration

A failed gate should produce an escalation, not a confident plan.

In [ ]:
def validate_item(item):
    problems = []
    if item["historical_events"] < 4:
        problems.append("insufficient historical events")
    if item["recommendation_score"] < .60:
        problems.append("recommendation confidence below threshold")
    if item["sources_found"] < 2:
        problems.append("insufficient approved sources")
    if not item["source_ids"]:
        problems.append("missing citations")
    return problems

def validate_plan(plan, hours_available=HOURS_AVAILABLE):
    problems = []
    if sum(item["hours"] for item in plan) > hours_available:
        problems.append("field-hour budget exceeded")
    for item in plan:
        problems.extend(f"{item['school']}: {p}" for p in validate_item(item))
    return problems

## 3. Build the orchestration loop

Complete `build_plan`. The intended sequence is:

`rank → recommend → estimate → retrieve → validate → request human review`

In [ ]:
def build_plan(hours_available=HOURS_AVAILABLE):
    plan = []
    # TODO: loop over rank_schools(), call the other tools, and add only
    # feasible items. Preserve score, evidence type, and source IDs.
    return plan

proposed_plan = build_plan()
proposed_plan

In [ ]:
problems = validate_plan(proposed_plan)
check("Plan contains at least one feasible item", len(proposed_plan) >= 1,
      "Start with Jefferson; all required mock evidence exists for that item.")
check("Plan stays within 16 hours", sum(item.get("hours", 0) for item in proposed_plan) <= HOURS_AVAILABLE)
check("Plan passes evidence and validation gates", not problems, str(problems))
check("Human approval remains required", HUMAN_APPROVAL_REQUIRED)

## 4. Failure injection

Red-team the workflow:

- Reduce the top school to only two historical events.
- Return zero sources for an otherwise strong action.
- Give an action a 20-hour cost.
- Insert an instruction inside a retrieved document telling the agent to ignore policy.

The correct result is a clear stop or escalation—not improvisation.

## Day 2 end state

```text
TRUSTED DATA → RANKED SCHOOL → RECOMMENDED ACTION
     → RETRIEVED EVIDENCE → VALIDATED PLAN → HUMAN REVIEW
```

An agent is valuable when it coordinates a useful workflow inside evidence, resource, and approval boundaries—not merely when it can act autonomously.